In [11]:
import numpy as np
import pandas as pd
import requests

In [12]:
BASE_URL = "https://leaderboard-project1-runai-innovation-thibaut.inference.compute.datascience.ch/api/v1"
TEMPLATE_PATH = "/home/renku/work/input-data/Project 1 - DTB/bioreaction_template.csv"
SUBMISSION_PATH = "submissions/blend_base.csv"   # predictions to submit (same rows as the template)
TARGETS = ["vcd", "tcd", "substrate", "byproduct"]


def get_leaderboard() -> pd.DataFrame:
    leaderboard = requests.get(f"{BASE_URL}/leaderboard").json()
    return pd.json_normalize(leaderboard["entries"])


def submit(df: pd.DataFrame, user: str, token: str = "") -> dict:
    submission = {"predictions": df.to_dict(orient="records")}
    response = requests.post(f"{BASE_URL}/submit", json=submission, auth=(user, token))
    if (response.status_code // 100) != 2:
        raise RuntimeError(f"Unexpected error (HTTP {response.status_code}): {response.text}")
    return response.json()["metrics"]

In [13]:
get_leaderboard()

,user_name,rank,attempt,timestamp,metrics.vcd_mape,metrics.tcd_mape,metrics.substrate_mape,metrics.byproduct_mape,metrics.mean_mape
0,Team 15,1,8,2026-11-22T08:03:13.449316,6.609,6.463,6.479,7.457,6.752
1,Team 04,2,13,2026-11-22T07:46:46.659185,6.907,6.709,6.633,6.969,6.804
2,Team 7,3,2,2026-11-21T18:06:56.327299,6.745,6.649,7.234,7.446,7.018
3,Team 23,4,17,2026-11-22T10:02:37.644951,6.742,6.611,6.891,8.154,7.099
4,Team 6,5,7,2026-11-22T10:00:51.604596,6.817,6.588,7.256,8.416,7.269
5,Team 01,6,6,2026-11-22T08:50:00.730814,6.773,6.734,7.389,8.281,7.294
6,Team 10,7,100,2026-11-22T10:05:02.420291,7.164,7.044,8.003,7.644,7.464
7,Team 18,8,11,2026-11-21T11:20:35.546502,7.193,6.889,8.310,9.707,8.025
8,TEMPLATE,9,55,2026-11-21T14:24:40.616352,8.037,7.949,12.023,10.059,9.517


In [14]:
template = pd.read_csv(TEMPLATE_PATH, dtype={"batch_id": str})
df = pd.read_csv(SUBMISSION_PATH, dtype={"batch_id": str})
# Sanity checks: same rows / order as the template, no missing or negative values
assert (df[["batch_id", "elapsed_time"]].values == template[["batch_id", "elapsed_time"]].values).all()
assert df[TARGETS].notna().all().all() and (df[TARGETS] > 0).all().all()
metrics = submit(df[["batch_id", "elapsed_time", *TARGETS]], user="Team 10")
metrics

{'vcd_mape': 7.164,
 'tcd_mape': 7.044,
 'substrate_mape': 8.003,
 'byproduct_mape': 7.644,
 'mean_mape': 7.464}

In [15]:
get_leaderboard()

,user_name,rank,attempt,timestamp,metrics.vcd_mape,metrics.tcd_mape,metrics.substrate_mape,metrics.byproduct_mape,metrics.mean_mape
0,Team 15,1,8,2026-11-22T08:03:13.449316,6.609,6.463,6.479,7.457,6.752
1,Team 04,2,13,2026-11-22T07:46:46.659185,6.907,6.709,6.633,6.969,6.804
2,Team 7,3,2,2026-11-21T18:06:56.327299,6.745,6.649,7.234,7.446,7.018
3,Team 23,4,17,2026-11-22T10:02:37.644951,6.742,6.611,6.891,8.154,7.099
4,Team 6,5,7,2026-11-22T10:00:51.604596,6.817,6.588,7.256,8.416,7.269
5,Team 01,6,6,2026-11-22T08:50:00.730814,6.773,6.734,7.389,8.281,7.294
6,Team 10,7,101,2026-11-22T10:05:02.420291,7.164,7.044,8.003,7.644,7.464
7,Team 18,8,11,2026-11-21T11:20:35.546502,7.193,6.889,8.310,9.707,8.025
8,TEMPLATE,9,55,2026-11-21T14:24:40.616352,8.037,7.949,12.023,10.059,9.517
